# 06 Sensitivity to the purchase value (ML7)

Owner: Jeevan Mandala (ML Engineer)

Does it matter whether a matrix cell holds the raw quantity, a 1/0 flag or log(1 + quantity), which is what we used? This is a model choice, so we compare it on the validation period from the training rows and never on the test purchases. k is the one from notebook 02.

In [4]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path

import numpy as np
import pandas as pd

from split import time_split
from matrix import build_matrix, restrict_eval_pairs
from recommend import (PROCESSED, fit_svd, product_vectors, recommend_for_customers, recommend_blend,
                       best_seller_baseline, hits_at_n, bootstrap_ci)

K = int(Path("../outputs/results/chosen_k.txt").read_text())
train = pd.read_pickle(PROCESSED / "train_transactions.pkl")
inner_train, _, val_held_out, _ = time_split(train, months=2)
print("k =", K)

k = 100


In [5]:
rows = []
recon_hits = {}
for value in ["log", "count", "binary"]:
    A_v, customers_v, products_v, info_v = build_matrix(inner_train, 10, value=value)
    pairs_v, _ = restrict_eval_pairs(val_held_out, customers_v, products_v)
    eval_rows_v = np.sort(pairs_v["customer_idx"].unique())
    svd_v = fit_svd(A_v, K)

    recon = hits_at_n(recommend_blend(A_v, svd_v, eval_rows_v, mode="reconstruct", alpha=1.0),
                      eval_rows_v, pairs_v, A_v.shape)
    cosine = hits_at_n(recommend_for_customers(A_v, product_vectors(svd_v), eval_rows_v),
                       eval_rows_v, pairs_v, A_v.shape)
    base = hits_at_n(best_seller_baseline(A_v, eval_rows_v), eval_rows_v, pairs_v, A_v.shape)
    recon_hits[value] = recon
    low, high = bootstrap_ci(recon)
    rows.append({
        "purchase value": info_v["purchase_value"],
        "reconstruct_hit_rate": recon.mean(), "ci_low": low, "ci_high": high,
        "cosine_hit_rate": cosine.mean(),
        "baseline_hit_rate": base.mean(),
    })
sensitivity = pd.DataFrame(rows).round(4)
sensitivity.to_csv("../outputs/results/purchase_value_sensitivity.csv", index=False)
sensitivity

,purchase value,reconstruct_hit_rate,ci_low,ci_high,cosine_hit_rate,baseline_hit_rate
0,log1p(total quantity per customer and product),0.3901,0.3654,0.4148,0.2269,0.2643
1,total quantity per customer and product (raw),0.2311,0.2099,0.2530,0.1322,0.2643
2,1 if the customer ever bought the product,0.4177,0.3908,0.4445,0.2290,0.2643


In [6]:
# the three matrices keep the same customers and products, so the evaluation customers
# are identical and we can compare the hit rates directly
from recommend import paired_bootstrap_diff

assert len({len(h) for h in recon_hits.values()}) == 1
paired_rows = []
for name in ("count", "binary"):
    diff, low, high = paired_bootstrap_diff(recon_hits[name], recon_hits["log"])
    paired_rows.append({"purchase value": name, "hit_rate_minus_log": diff, "ci_low": low, "ci_high": high})
paired_values = pd.DataFrame(paired_rows).round(4)
paired_values.to_csv("../outputs/results/purchase_value_paired.csv", index=False)
paired_values

,purchase value,hit_rate_minus_log,ci_low,ci_high
0,count,-0.1590,-0.1845,-0.1314
1,binary,0.0276,0.0049,0.0509


## What this shows

Raw quantity is clearly the worst, 0.231 against 0.390 for the log. A few huge wholesale orders dominate the factors.

The 1/0 flag scored higher than the log, 0.418 against 0.390. The paired difference is +0.028 (95% interval 0.005 to 0.051), which is the fairer comparison because both are scored on the same customers, so the gain is probably real.

We still keep log1p(quantity). It was fixed in the data step, k and the scoring were chosen with it, and the test set has already been looked at once, so switching now would mean a second look at it. The 1/0 flag is a next step in the memo.

The baseline is the same in every row (0.2643) because it only counts buyers.